# VectorAlpha Computation Library EDA

VectorAlpha is intended for fast vectorized indicators and benchmarkable technical feature implementations. This notebook documents the target integration shape and shows the current environment status.

In [1]:
from __future__ import annotations

import importlib
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

def library_status(module_name: str) -> pd.DataFrame:
    try:
        module = importlib.import_module(module_name)
        return pd.DataFrame([{
            "module": module_name,
            "available": True,
            "version": getattr(module, "__version__", "unknown"),
            "module_file": getattr(module, "__file__", None),
        }])
    except Exception as exc:
        return pd.DataFrame([{
            "module": module_name,
            "available": False,
            "version": None,
            "module_file": None,
            "error": f"{type(exc).__name__}: {exc}",
        }])

def synthetic_prices(rows: int = 360, symbol: str = "AAPL") -> pd.DataFrame:
    rng = np.random.default_rng(7)
    dates = pd.date_range("2024-01-02", periods=rows, freq="B")
    drift = 0.00055
    shock = rng.normal(0, 0.015, rows)
    close = 100 * np.exp(np.cumsum(drift + shock))
    open_ = close * (1 + rng.normal(0, 0.003, rows))
    high = np.maximum(open_, close) * (1 + rng.uniform(0.001, 0.02, rows))
    low = np.minimum(open_, close) * (1 - rng.uniform(0.001, 0.02, rows))
    volume = rng.integers(900_000, 4_500_000, rows).astype(float)
    return pd.DataFrame({
        "symbol": symbol,
        "date": dates,
        "open": open_,
        "high": high,
        "low": low,
        "close": close,
        "volume": volume,
    })

prices = synthetic_prices()
prices.head()

,symbol,date,open,high,low,close,volume
0,AAPL,2024-01-02,100.2080,101.2355,98.8738,100.0569,"4,479,129.0000"
1,AAPL,2024-01-03,101.1259,102.2340,100.4497,100.5615,"2,365,734.0000"
2,AAPL,2024-01-04,100.3819,101.9042,99.3952,100.2040,"1,409,205.0000"
3,AAPL,2024-01-05,98.9452,99.8670,97.0239,98.9286,"4,225,356.0000"
4,AAPL,2024-01-08,97.8130,98.6722,97.4943,98.3103,"2,886,674.0000"


## Library Status

In [2]:
library_status("vectoralpha")

,module,available,version,module_file,error
0,vectoralpha,False,None,None,ModuleNotFoundError: No module named 'vectoral...


## Intended Feature Engineering Contract

When `vectoralpha` is available, its outputs should be stored as a separate computation-library family, not mixed into pandas-ta-classic outputs. The contract should look like a `BuiltFeatureSet`: MultiIndex `(date, symbol)`, prefixed feature columns, and no target leakage.

In [3]:
from quant_warehouse.feature_engineering import BuiltFeatureSet

# Contract-only example: this is the shape VectorAlpha adapters should return.
work = prices.set_index("date")
returns = work["close"].pct_change()
contract_frame = pd.DataFrame({
    "va__ret_1d": returns,
    "va__sma_20": work["close"].rolling(20).mean(),
    "va__dist_sma_20": work["close"] / work["close"].rolling(20).mean() - 1,
    "va__vol_20": returns.rolling(20).std(),
})
contract_frame["symbol"] = "AAPL"
contract_frame = contract_frame.reset_index().set_index(["date", "symbol"]).sort_index()
built = BuiltFeatureSet(df=contract_frame.dropna(), feature_cols=[c for c in contract_frame.columns if c.startswith("va__")])
{"rows": len(built.df), "feature_count": len(built.feature_cols), "sample_features": built.feature_cols}

{'rows': 340,
 'feature_count': 4,
 'sample_features': ['va__ret_1d',
  'va__sma_20',
  'va__dist_sma_20',
  'va__vol_20']}

In [4]:
built.df.tail()

,,va__ret_1d,va__sma_20,va__dist_sma_20,va__vol_20
date,symbol,,,,
2025-05-13,AAPL,0.0050,69.1183,0.0126,0.0141
2025-05-14,AAPL,-0.0039,69.2983,0.0060,0.0137
2025-05-15,AAPL,-0.0001,69.5209,0.0028,0.0132
2025-05-16,AAPL,0.0037,69.7010,0.0038,0.0128
2025-05-19,AAPL,-0.0007,69.8001,0.0017,0.0118


## Benchmarking Against Existing Modules

The reason to add VectorAlpha is speed and implementation diversity. It should be benchmarked against `compute_features_worldclass` and `pandas_ta_classic` for the same symbol universe and horizons.

In [5]:
from quant_warehouse.feature_engineering import build_price_technical_features

base_features = build_price_technical_features("AAPL", prices)
comparison = pd.DataFrame([
    {"family": "worldclass_pandas", "rows": len(base_features.df), "features": len(base_features.feature_cols)},
    {"family": "vectoralpha_contract", "rows": len(built.df), "features": len(built.feature_cols)},
])
comparison

,family,rows,features
0,worldclass_pandas,360,122
1,vectoralpha_contract,340,4


## Target Engineering Connection

VectorAlpha features can be joined to forward-return or rank labels. Keep the library choice in feature metadata so future experiments can compare `va__sma_20` versus `ta_overlap__sma_20` versus custom CUDA indicators.

In [6]:
from quant_warehouse.target_engineering import build_forward_return_labels

labels = build_forward_return_labels(prices[["symbol", "date", "close"]], horizons=[5, 20])
joined = built.df.reset_index().merge(labels, on=["date", "symbol"], how="inner")
joined.tail()

,date,symbol,va__ret_1d,va__sma_20,va__dist_sma_20,va__vol_20,horizon,target_name,target_value
675,2025-05-15,AAPL,-0.0001,69.5209,0.0028,0.0132,20,forward_return_20d,NaN
676,2025-05-16,AAPL,0.0037,69.7010,0.0038,0.0128,5,forward_return_5d,NaN
677,2025-05-16,AAPL,0.0037,69.7010,0.0038,0.0128,20,forward_return_20d,NaN
678,2025-05-19,AAPL,-0.0007,69.8001,0.0017,0.0118,5,forward_return_5d,NaN
679,2025-05-19,AAPL,-0.0007,69.8001,0.0017,0.0118,20,forward_return_20d,NaN


## Notes

- `vectoralpha` is not installed in this environment, so this notebook does not claim runtime VectorAlpha results.
- The adapter should live under `platforms/computation_libraries/vectoralpha/` and expose reusable functions that return `BuiltFeatureSet`.
- Do not replace existing feature engineering modules; add VectorAlpha as a benchmarkable implementation choice.